In [1]:
!nvidia-smi

%cd /content
!rm -rf CIRI-FS
!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git
%cd /content/CIRI-FS

!git branch --show-current

Wed Jul 15 21:24:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import pandas as pd
import torch
import transformers
import bitsandbytes

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

numpy: 2.5.1
pandas: 2.2.2
torch: 2.11.0+cu128
transformers: 5.14.0
bitsandbytes: 0.49.2


In [3]:
!pip install -q -r requirements.txt

In [4]:
import torch
import transformers
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None"
)

PyTorch: 2.11.0+cu128
Transformers: 5.14.0
CUDA available: True
GPU: Tesla T4


In [5]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully


In [6]:
from pathlib import Path

path = Path("ciri/query/llm_gen.py")
text = path.read_text()

marker = "class QwenGen(BaseGen):"

new_qwen_class = r'''class QwenGen(BaseGen):
    def __init__(self, args: Dict, config_file: str, model, tokenizer):
        super().__init__(args, config_file)

        self.llm_model = model
        self.tokenizer = tokenizer
        self.device = get_device()

        # Cost measurements for the current configuration file
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_generation_time = 0.0
        self.generation_calls = 0

    def _generate(self) -> List:
        message = f"{self.config_file}\n{self.prompt}"

        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": message
            }
        ]

        formatted = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            formatted,
            return_tensors="pt"
        ).to(self.device)

        input_len = inputs["input_ids"].shape[1]

        # Ensure GPU timing is accurate
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        with torch.inference_mode():
            outputs = self.llm_model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.2,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

        answer_list = []
        output_token_count = 0

        for output in outputs:
            generated_tokens = output[input_len:]
            output_token_count += generated_tokens.numel()

            answer = self.tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            answer_list.append(answer)

        self.generation_calls += 1
        self.total_input_tokens += input_len
        self.total_output_tokens += output_token_count
        self.total_generation_time += elapsed_time

        logger.info(
            "[Qwen Cost] "
            f"call={self.generation_calls}, "
            f"input_tokens={input_len}, "
            f"output_tokens={output_token_count}, "
            f"total_tokens={input_len + output_token_count}, "
            f"generation_time_seconds={elapsed_time:.4f}"
        )

        logger.info(
            "[Qwen Cost Cumulative] "
            f"calls={self.generation_calls}, "
            f"input_tokens={self.total_input_tokens}, "
            f"output_tokens={self.total_output_tokens}, "
            f"total_tokens="
            f"{self.total_input_tokens + self.total_output_tokens}, "
            f"generation_time_seconds="
            f"{self.total_generation_time:.4f}"
        )

        return answer_list
'''

if marker in text:
    # QwenGen is currently the last class in llm_gen.py.
    text_before_qwen = text.split(marker, 1)[0]
    path.write_text(text_before_qwen + new_qwen_class + "\n")
    print("Existing QwenGen replaced.")
else:
    path.write_text(text.rstrip() + "\n\n" + new_qwen_class + "\n")
    print("New QwenGen added.")

Existing QwenGen replaced.


In [ ]:
!python -m py_compile ciri/query/llm_gen.py

In [7]:
from pathlib import Path

path = Path("ciri/ciri_runner.py")
text = path.read_text()

old_import = (
    "from ciri.query.llm_gen import "
    "GPTGen, ClaudeGen, LlamaGen, DeepseekGen"
)

new_import = (
    "from ciri.query.llm_gen import "
    "GPTGen, ClaudeGen, LlamaGen, DeepseekGen, QwenGen"
)

if old_import in text:
    text = text.replace(old_import, new_import)

if 'args.model.startswith("Qwen")' not in text:
    old_block = '''    elif args.model.startswith("deepseek"):
        return DeepseekGen(args, file_content, model, tokenizer)
    else:'''

    new_block = '''    elif args.model.startswith("deepseek"):
        return DeepseekGen(args, file_content, model, tokenizer)
    elif args.model.startswith("Qwen"):
        return QwenGen(args, file_content, model, tokenizer)
    else:'''

    if old_block not in text:
        raise RuntimeError(
            "Could not locate the model-selection block "
            "in ciri_runner.py."
        )

    text = text.replace(old_block, new_block)

path.write_text(text)

print("ciri_runner.py is ready.")

ciri_runner.py is ready.


In [8]:
from pathlib import Path
import re

path = Path("ciri/ciri_eng.py")
text = path.read_text()

# Add Qwen to the command-line model choices.
if '"Qwen2.5-Coder-7B-Instruct"' not in text:
    text, count = re.subn(
        r'("deepseek-coder-6\.7b-instruct")',
        r'\1, "Qwen2.5-Coder-7B-Instruct"',
        text,
        count=1
    )

    if count == 0:
        raise RuntimeError(
            "Could not add Qwen to the model choices."
        )

# Add the Qwen loading branch.
marker = '        elif checkpoint.startswith("CodeLLaMa"):'

qwen_loader = '''        elif checkpoint.startswith("Qwen"):
            from transformers import BitsAndBytesConfig

            full_checkpoint = (
                "Qwen/Qwen2.5-Coder-7B-Instruct"
            )

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )

            model = AutoModelForCausalLM.from_pretrained(
                full_checkpoint,
                quantization_config=quantization_config,
                device_map="auto",
                trust_remote_code=True
            )

            tokenizer = AutoTokenizer.from_pretrained(
                full_checkpoint,
                trust_remote_code=True,
                padding_side="left"
            )

            if tokenizer.pad_token_id is None:
                tokenizer.pad_token = tokenizer.eos_token

'''

if 'checkpoint.startswith("Qwen")' not in text:
    if marker not in text:
        raise RuntimeError(
            "Could not locate the CodeLLaMa loader block."
        )

    text = text.replace(marker, qwen_loader + marker)

# Quantized models with device_map="auto" must not be moved again.
text = text.replace(
    '        model = model.to("cuda")\n',
    ''
)

path.write_text(text)

print("ciri_eng.py is ready.")

ciri_eng.py is ready.


In [9]:
!python -m py_compile \
    ciri/query/llm_gen.py \
    ciri/ciri_runner.py \
    ciri/ciri_eng.py

In [10]:
!python -m ciri.ciri_eng --help

usage: ciri_eng.py [-h] --input_path INPUT_PATH --output_path OUTPUT_PATH
                   --model
                   {gpt-3.5-turbo-0125,gpt-4-0125-preview,claude-3-opus-20240228,claude-3-sonnet-20240229,CodeLLaMa-7b-Instruct-hf,CodeLLaMa-13b-Instruct-hf,CodeLLaMa-34b-Instruct-hf,deepseek-coder-6.7b-instruct,Qwen2.5-Coder-7B-Instruct}
                   --system SYSTEM --version VERSION
                   [--validconfig_shot_num {0,1,2,3,4,5}]
                   [--misconfig_shot_num {0,1,2,3,4,5}]
                   [--shot_selection {random,similarity}]
                   [--file_format {xml,yaml,properties,conf}]
                   [--language {java,python,cpp}] [--read_code]
                   [--read_code_loc READ_CODE_LOC] [--shot_system SHOT_SYSTEM]
                   [--verbose]

Ciri Engine - Configuration Analysis Tool

options:
  -h, --help            show this help message and exit
  --validconfig_shot_num {0,1,2,3,4,5}
                        Number of valid configurati

In [11]:
!find \
    icse25_data/datasets/synthesize_config/ext4_SD \
    -maxdepth 2 \
    -type f \
    | sort

icse25_data/datasets/synthesize_config/ext4_SD/correct/1
icse25_data/datasets/synthesize_config/ext4_SD/correct/2
icse25_data/datasets/synthesize_config/ext4_SD/correct/3
icse25_data/datasets/synthesize_config/ext4_SD/correct/4
icse25_data/datasets/synthesize_config/ext4_SD/correct/5
icse25_data/datasets/synthesize_config/ext4_SD/erroneous/1
icse25_data/datasets/synthesize_config/ext4_SD/erroneous/2
icse25_data/datasets/synthesize_config/ext4_SD/erroneous/3
icse25_data/datasets/synthesize_config/ext4_SD/erroneous/4
icse25_data/datasets/synthesize_config/ext4_SD/erroneous/5


In [12]:
!cat \
    icse25_data/datasets/synthesize_config/ground_truth/ext4_SD.tsv

Range	Basic Numeric	s_first_meta_bg	99999999
Range	Basic Numeric	e2fsck.discard_end_group	-1
Range	Basic Numeric	mke2fs.blocksize	999999
Range	Basic Numeric	ext4.s_log_block_size	-1
Range	Basic Numeric	ext4.s_first_meta_bg	65536


In [13]:
!rm -rf \
    icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct

In [14]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/erroneous \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 0 \
    --misconfig_shot_num 0 \
    --file_format xml \
    --verbose

2026-07-15 21:39:48 - Ciri - INFO - Using device: CUDA
2026-07-15 21:39:48 - Ciri - INFO - Using dtype: torch.bfloat16
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 49.1MB/s]
Fetching 4 files: 100% 4/4 [08:40<00:00, 130.11s/it]
Download complete: 100% 15.2G/15.2G [08:40<00:00, 41.8MB/s]                
Loading weights:   0% 0/339 [00:00<?, ?it/s]
Loading weights:   0% 1/339 [00:04<25:30,  4.53s/it]
Loading weights:   1% 2/339 [00:08<24:59,  4.45s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Download complete: 100% 15.2G/15.2G [08:51<00:00, 41.8MB/s]
Loading weights:   1% 5/339 [00:10<08:07,  1.46s/it]
Loading weights:   2% 6/339 [00:10<06:33,  1.18s/it]
Loading weights:   3% 10/339 [00:10<02:27,  2.23it/s]
Loading weights:   4% 12/339 [00:10<01:45,  3.09it/s]
Load

In [15]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot/correct \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 0 \
    --misconfig_shot_num 0 \
    --file_format xml \
    --verbose

2026-07-15 21:51:27 - Ciri - INFO - Using device: CUDA
2026-07-15 21:51:27 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights:   1% 2/339 [00:08<24:28,  4.36s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:01<00:00,  5.51it/s]
2026-07-15 21:52:40 - Ciri - INFO - Model loaded successfully on CUDA!
2026-07-15 21:52:40 - Ciri - INFO - [llm_gen] Using device: CUDA
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
2026-07-15 21:52:44 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=295, output_tokens=25, total_tokens=320, generation_time_seconds=

In [16]:
!python icse25_data/script/result_parser.py \
    --project ext4_SD \
    --model Qwen2.5-Coder-7B-Instruct \
    --mode zero_shot

[Ciri Result] on ext4_SD with Qwen2.5-Coder-7B-Instruct and zero_shot mode
File-Level: Precision: 1.00, Recall: 0.20, Accuracy: 0.60, F1: 0.33
Param-Level: Precision: 1.00, Recall: 0.20, Accuracy: 0.95, F1: 0.33


In [18]:
!grep -c "\[Qwen Cost\]" ciri_debug.log

0


In [20]:
!grep -R "\[Qwen Cost\]" \
  icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/zero_shot

In [21]:
import pandas as pd

summary = pd.DataFrame([{
    "model": "Qwen2.5-Coder-7B-Instruct",
    "benchmark": "EXT4_SD",
    "prompting": "zero_shot",
    "model_calls": 30,
    "input_tokens": 9210,
    "output_tokens": 903,
    "total_tokens": 10113,
    "generation_seconds": 97.33,
    "file_precision": 1.00,
    "file_recall": 0.20,
    "file_accuracy": 0.60,
    "file_f1": 0.33,
    "parameter_precision": 1.00,
    "parameter_recall": 0.20,
    "parameter_accuracy": 0.95,
    "parameter_f1": 0.33,
}])

summary.to_csv("qwen_ext4_sd_zero_shot_summary.csv", index=False)
summary

,model,benchmark,prompting,model_calls,input_tokens,output_tokens,total_tokens,generation_seconds,file_precision,file_recall,file_accuracy,file_f1,parameter_precision,parameter_recall,parameter_accuracy,parameter_f1
0,Qwen2.5-Coder-7B-Instruct,EXT4_SD,zero_shot,30,9210,903,10113,97.33,1.0,0.2,0.6,0.33,1.0,0.2,0.95,0.33
